In [55]:
import pandas as pd
pd.options.mode.chained_assignment = None
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
%matplotlib inline 
import matplotlib.pyplot as plt

In [56]:
patients_data = pd.read_csv("Texture features.csv", index_col=0)

In [57]:
patients_data


,INFO_PatientID,INFO_ProcessDateOfTexture,INFO_SeriesDate,INFO_Series,INFO_ActualFrameDuration,INFO_NameOfRoi,CONVENTIONAL_SUVbwmin,CONVENTIONAL_SUVbwmean,CONVENTIONAL_SUVbwstd,CONVENTIONAL_SUVbwmax,...,GLZLM_HGZE,GLZLM_SZLGE,GLZLM_SZHGE,GLZLM_LZLGE,GLZLM_LZHGE,GLZLM_GLNU,GLZLM_ZLNU,GLZLM_ZP,TimePosition,zLocation[onlyFor2DROI]
INFO_PatientName,,,,,,,,,,,,,,,,,,,,,
LIFEX,LIFEx,Sun Jan 23 18:19:19 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.108004,0.319234,0.064479,0.476722,...,2.500000,0.000192,0.000585,2107.625000,11034.500000,1.000000,1.000000,0.016129,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:19:20 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,2.910082,5.854704,1.387413,8.991810,...,363.808824,0.002428,208.152420,0.021055,3025.955882,4.323529,25.000000,0.482270,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:38:58 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.188521,0.323404,0.057192,0.449061,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:38:59 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,3.238746,5.991053,1.307934,8.991810,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:51:51 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.188521,0.323404,0.057192,0.449061,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:51:52 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,3.238746,5.991053,1.307934,8.991810,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 19:07:19 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.191164,0.329256,0.058172,0.474525,...,2.500000,0.000357,0.001024,1151.125000,6418.000000,1.000000,1.000000,0.021505,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 19:07:20 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,3.138171,5.955576,1.347132,8.987144,...,366.479167,0.002451,211.388504,0.019289,3109.687500,3.458333,18.666667,0.480000,0.0,NaN


In [58]:
numerics = ['int16','int32','int64','float16','float32','float64']
numerical_vars = list(patients_data.select_dtypes(include=numerics).columns)
patients_data = patients_data[numerical_vars]
patients_data

,INFO_NameOfRoi,CONVENTIONAL_SUVbwmin,CONVENTIONAL_SUVbwmean,CONVENTIONAL_SUVbwstd,CONVENTIONAL_SUVbwmax,CONVENTIONAL_SUVbwQ1,CONVENTIONAL_SUVbwQ2,CONVENTIONAL_SUVbwQ3,CONVENTIONAL_SUVbwSkewness,CONVENTIONAL_SUVbwKurtosis,...,GLZLM_HGZE,GLZLM_SZLGE,GLZLM_SZHGE,GLZLM_LZLGE,GLZLM_LZHGE,GLZLM_GLNU,GLZLM_ZLNU,GLZLM_ZP,TimePosition,zLocation[onlyFor2DROI]
INFO_PatientName,,,,,,,,,,,,,,,,,,,,,
LIFEX,0.0,0.108004,0.319234,0.064479,0.476722,0.283809,0.321648,0.372574,-0.502561,3.427661,...,2.500000,0.000192,0.000585,2107.625000,11034.500000,1.000000,1.000000,0.016129,0.0,NaN
LIFEX,1.0,2.910082,5.854704,1.387413,8.991810,4.692568,5.716900,6.903404,0.116157,2.321317,...,363.808824,0.002428,208.152420,0.021055,3025.955882,4.323529,25.000000,0.482270,0.0,NaN
LIFEX,0.0,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,1.0,3.238746,5.991053,1.307934,8.991810,5.146196,5.923580,6.928142,0.150938,2.437242,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,0.0,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,1.0,3.238746,5.991053,1.307934,8.991810,5.146196,5.923580,6.928142,0.150938,2.437242,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,0.0,0.191164,0.329256,0.058172,0.474525,0.290586,0.332372,0.375901,-0.145099,2.462997,...,2.500000,0.000357,0.001024,1151.125000,6418.000000,1.000000,1.000000,0.021505,0.0,NaN
LIFEX,1.0,3.138171,5.955576,1.347132,8.987144,5.040182,5.886095,7.073061,0.107705,2.348289,...,366.479167,0.002451,211.388504,0.019289,3109.687500,3.458333,18.666667,0.480000,0.0,NaN


In [59]:
X = patients_data.drop(['INFO_NameOfRoi','zLocation[onlyFor2DROI]','CONVENTIONAL_SUVbwcalciumAgatstonScore[onlyForCT]','CONVENTIONAL_SUVbwpeakSphere1mL[onlyFor3D]','CONVENTIONAL_SUVbwpeakSphere1mL:discretized volume sought[onlyFor3D]','CONVENTIONAL_SUVbwpeakSphere0.5mL[onlyFor3D]','CONVENTIONAL_SUVbwpeakSphere0.5mL:discretized volume sought[onlyFor3D]'], axis=1)
X

,CONVENTIONAL_SUVbwmin,CONVENTIONAL_SUVbwmean,CONVENTIONAL_SUVbwstd,CONVENTIONAL_SUVbwmax,CONVENTIONAL_SUVbwQ1,CONVENTIONAL_SUVbwQ2,CONVENTIONAL_SUVbwQ3,CONVENTIONAL_SUVbwSkewness,CONVENTIONAL_SUVbwKurtosis,CONVENTIONAL_SUVbwExcessKurtosis,...,GLZLM_LGZE,GLZLM_HGZE,GLZLM_SZLGE,GLZLM_SZHGE,GLZLM_LZLGE,GLZLM_LZHGE,GLZLM_GLNU,GLZLM_ZLNU,GLZLM_ZP,TimePosition
INFO_PatientName,,,,,,,,,,,,,,,,,,,,,
LIFEX,0.108004,0.319234,0.064479,0.476722,0.283809,0.321648,0.372574,-0.502561,3.427661,0.427661,...,0.625000,2.500000,0.000192,0.000585,2107.625000,11034.500000,1.000000,1.000000,0.016129,0.0
LIFEX,2.910082,5.854704,1.387413,8.991810,4.692568,5.716900,6.903404,0.116157,2.321317,-0.678683,...,0.003572,363.808824,0.002428,208.152420,0.021055,3025.955882,4.323529,25.000000,0.482270,0.0
LIFEX,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,-0.650819,...,0.750000,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0
LIFEX,3.238746,5.991053,1.307934,8.991810,5.146196,5.923580,6.928142,0.150938,2.437242,-0.562758,...,0.003126,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0
LIFEX,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,-0.650819,...,0.750000,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0
LIFEX,3.238746,5.991053,1.307934,8.991810,5.146196,5.923580,6.928142,0.150938,2.437242,-0.562758,...,0.003126,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0
LIFEX,0.191164,0.329256,0.058172,0.474525,0.290586,0.332372,0.375901,-0.145099,2.462997,-0.537003,...,0.625000,2.500000,0.000357,0.001024,1151.125000,6418.000000,1.000000,1.000000,0.021505,0.0
LIFEX,3.138171,5.955576,1.347132,8.987144,5.040182,5.886095,7.073061,0.107705,2.348289,-0.651711,...,0.003468,366.479167,0.002451,211.388504,0.019289,3109.687500,3.458333,18.666667,0.480000,0.0


In [60]:
X.isnull().sum()

CONVENTIONAL_SUVbwmin     0
CONVENTIONAL_SUVbwmean    0
CONVENTIONAL_SUVbwstd     0
CONVENTIONAL_SUVbwmax     0
CONVENTIONAL_SUVbwQ1      0
                         ..
GLZLM_LZHGE               0
GLZLM_GLNU                0
GLZLM_ZLNU                0
GLZLM_ZP                  0
TimePosition              0
Length: 74, dtype: int64

In [61]:
y = np.asarray(patients_data[['INFO_NameOfRoi']])
y

array([[0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.]])

In [62]:
#Chi-Squared feature selection is done as follows
#Firstly, the libraries are imported
from sklearn import datasets
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

In [65]:
X = patients_data.drop('GLZLM_LZLGE',axis=1)
y = patients_data['GLZLM_LZLGE']
chi_scores = chi2(X,y)


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [64]:
chi_scores

NameError: name 'chi_scores' is not defined